In [1]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

!pip install selenium beautifulsoup4
!pip install webdriver-manager


OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,632 B in 2s (1,830 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), 

In [2]:
import time
import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager



In [3]:
from bs4 import BeautifulSoup
import re

def extract_stock_data(html_data):
  """
  HTMLを解析して株価データ（日付・始値・高値・安値・終値）を抽出する。
  引数：
      html_data: 解析対象のHTML文字列
  戻り値：
      list: [日付(YYYY-MM-DD), 始値, 高値, 安値, 終値] のリスト。
            データが取得できない場合は空リストを返す。
  """
  # html解析情報を生成する
  soup = BeautifulSoup(html_data, 'html.parser')

  try:
    # ─── 日付の取得 ───────────────────────────────────────────────
    # 候補セレクタ（日経サイトのDOM構造に合わせて順に試みる）
    date_selectors = [
      'div[class^="Chart_container"]', # Chart_container で始まるクラスを持つdiv
      'div[class*="ChartHeader_container"]',
      'div[class*="ChartHeader_row"]',
      'div[class*="ChartHeader_date"]',
    ]
    date_text = None
    for sel in date_selectors:
      el = soup.select_one(sel)
      # print(f"[DEBUG extract_stock_data] {el}")
      if el and el.get_text(strip=True):
        date_text = el.get_text(strip=True)
        break

    if not date_text:
      # print("[DEBUG extract_stock_data] 日付テキストが見つかりませんでした。")
      return []
    # print(f"[DEBUG extract_stock_data] 取得した日付テキスト: {date_text}")

    # 「yyyy年m月d日」「yyyy/m/d」「yyyy-m-d」などを YYYY-MM-DD に変換
    date_pattern = re.search(
      r'(\d{4})[年/\-](\d{1,2})[月/\-](\d{1,2})',
      date_text
    )
    if not date_pattern:
      # print("[DEBUG extract_stock_data] 日付パターンがマッチしませんでした。")
      return []
    year  = date_pattern.group(1)
    month = date_pattern.group(2).zfill(2)  # 0埋め
    day   = date_pattern.group(3).zfill(2)  # 0埋め
    formatted_date = f'{year}-{month}-{day}'

    # ─── 株価（始値・高値・安値・終値）の取得 ──────────────────────
    # 候補セレクタ
    value_selectors = [
      'span.m-chartData__value',
      'span[class*="value"]',
      'dd[class*="value"]',
      'td[class*="value"]',
    ]
    values = []
    for sel in value_selectors:
      elements = soup.select(sel)
      if elements:
        for el in elements:
          text = el.get_text(strip=True).replace(',', '')  # カンマ除去
          try:
            values.append(float(text))
          except ValueError:
            pass
        if len(values) >= 4:
          break

    # 始値・高値・安値・終値の4値が揃っているか確認
    if len(values) < 4:
      # print(f"[DEBUG extract_stock_data] 取得できた株価の数: {len(values)} (4つ未満)")
      return []
    # print(f"[DEBUG extract_stock_data] 取得した株価: {values}")

    open_price  = values[0]  # 始値
    high_price  = values[1]  # 高値
    low_price   = values[2]  # 安値
    close_price = values[3]  # 終値

    return [formatted_date, open_price, high_price, low_price, close_price]

  except Exception as e:
    print(f'[extract_stock_data] エラー: {e}')
    return []

In [4]:
def get_stock_values(driver, url):
  """
  Selenium WebDriver を使ってグラフ上をホバーし、株価データを収集する。
  グラフの右端からスタートし、1px ずつ左にずらしながら
  extract_stock_data() でデータを取得する。
  日付が変わった（＝新しい営業日データ）ときのみ記録する。
  引数：
    driver: Selenium WebDriver オブジェクト
    url: スクレイピング対象の URL
  戻り値：
    list[list]: [[日付, 始値, 高値, 安値, 終値], ...] の二次元リスト
  """
  # 最大30秒間、対象要素が表示されるのを待つ
  wait = WebDriverWait(driver, 30)

  # ページにアクセス
  driver.get(url)

  # iframeに切り替える
  iframe = driver.find_element(By.CSS_SELECTOR, 'iframe[title="iframe-chart"]')

  # 取得したiframeに切り替える
  driver.switch_to.frame(iframe)

  # ─── グラフ要素の特定 ────────────────────────────────────────────
  # 候補セレクタ（日経サイトのグラフ SVG / Canvas の親要素）
  chart_selectors = [
    'div[class^="Chart_container"]', # Chart_container で始まるクラスを持つdiv
    'div.highcharts-container',     # 最も一般的なHighchartsコンテナ
    'svg.highcharts-root',          # Highcharts SVGのルート要素
    'div[id^="highcharts-"]',      # 動的に生成されるHighcharts ID (例: highcharts-xxxx)
    'div.m-chartWrapper',
    'div[class*="chartWrapper"]',
    'div[class*="chart"] svg',
    'canvas',
    'rect.highcharts-plot-border'   # プロットエリアの境界線
  ]
  chart_el = None
  for sel in chart_selectors:
    try:
      # 要素がDOMに存在し、かつ表示されていることを待つ
      temp_el = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, sel)))
      # さらに、要素がゼロではないサイズを持つまで待つ
      wait.until(lambda d: temp_el.size['width'] > 0 and temp_el.size['height'] > 0)

      # 意味のある幅を持つ要素を採用
      if temp_el.size['width'] > 100: # ある程度の幅があるか確認
        chart_el = temp_el
        break # 適切な要素が見つかったらループを抜ける
    except Exception:
      continue

  if chart_el is None or chart_el.size['width'] == 0 or chart_el.size['height'] == 0:
    print('グラフ要素が見つかりませんでした、またはサイズが適切ではありませんでした。')
    return []

  chart_width  = chart_el.size['width']
  chart_height = chart_el.size['height']
  half_width   = chart_width // 2

  print(f'グラフサイズ: 幅={chart_width}px, 高さ={chart_height}px')

  actions = ActionChains(driver)

  # ─── グラフ中央 → 右端へ移動 ────────────────────────────────────
  # まずグラフ中央に移動し、そこから右に half_width だけずらして右端へ
  actions.move_to_element(chart_el).perform()           # 中央へ
  time.sleep(0.3)
  actions.move_by_offset(half_width, 0).perform()       # 右端へ
  time.sleep(0.5)

  stock_data   = []   # 収集したデータ
  seen_dates   = set() # 重複排除用

  # ─── 右端 → 左端へ 1px ずつスキャン ─────────────────────────────
  for _ in range(chart_width):
    # 現在のHTMLを取得して株価データを抽出
    html = driver.page_source
    row  = extract_stock_data(html)

    if row and row[0] not in seen_dates:  # 新しい日付のみ追加
      stock_data.append(row)
      seen_dates.add(row[0])

    # 1px 左へ
    actions.move_by_offset(-1, 0).perform()

  return stock_data

In [5]:
# ヘッドレスモードで起動するためのオプションを設定
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless=new') # ヘッドレスモードを有効にする (最新のheadlessモード)
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')         # GPU不使用
chrome_options.add_argument('--window-size=1920,1080') # ウィンドウサイズ指定
chrome_options.add_argument('--disable-blink-features=AutomationControlled') # Bot検知対策
chrome_options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/124.0.0.0 Safari/537.36'
)  # User-Agent を通常ブラウザに偽装


# ChromeDriverManagerを使用してChromeDriverを自動的に管理
service = Service(ChromeDriverManager().install())

# Google Chrome用のブラウザドライバーインスタンスを作成
chrome_driver = webdriver.Chrome(service=service, options=chrome_options)

# 開始日時を取得する
get_start_time = datetime.datetime.now()
# データ取得
data_list = get_stock_values(chrome_driver, 'https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month')
# 終了日時を取得する
get_end_time = datetime.datetime.now()
# 時間計測
execution_time = get_end_time - get_start_time
print(f"実行時間：{execution_time.total_seconds():.2f} 秒です")

try:
  # ─── 取得データの表示（日付昇順に並び替え） ───────────────────
  if data_list:
    # 日付昇順でソート
    data_list.sort(key=lambda x: x[0])

    header = f'{"日付":<7}  {"始値":>7}  {"高値":>7}  {"安値":>8}  {"終値":>8}'
    print(header)
    print('-' * 60)

    for row in data_list:
      date, open_p, high_p, low_p, close_p = row
      print(
        f'{date:<12}  '
        f'{open_p:>10,.2f}  '
        f'{high_p:>10,.2f}  '
        f'{low_p:>10,.2f}  '
        f'{close_p:>10,.2f}'
      )

    print('=' * 60)
    print(f'取得件数: {len(data_list)} 件')
  else:
    print('データが取得できませんでした。')
    print('ページ構造が変更された可能性があります。')
finally:
  # ブラウザを終了する
  chrome_driver.quit()


グラフサイズ: 幅=688px, 高さ=340px
実行時間：246.96 秒です
日付            始値       高値        安値        終値
------------------------------------------------------------
2026-02-25     57,695.40   58,875.17   57,656.50   58,583.12
2026-02-26     58,995.39   59,332.43   58,577.84   58,753.39
2026-02-27     58,606.03   58,924.17   58,130.57   58,850.27
2026-03-02     57,976.20   58,365.21   57,285.77   58,057.24
2026-03-03     57,729.80   57,890.76   56,091.54   56,279.05
2026-03-04     55,470.88   55,701.27   53,618.20   54,245.54
2026-03-05     55,204.16   56,619.98   54,910.33   55,278.06
2026-03-06     54,674.60   55,686.56   54,513.43   55,620.84
2026-03-09     54,608.63   54,608.63   51,407.66   52,728.72
2026-03-10     53,524.09   54,694.89   53,487.19   54,248.39
2026-03-11     54,917.93   55,745.38   54,882.58   55,025.37
2026-03-12     54,387.90   54,733.08   53,796.01   54,452.96
2026-03-13     53,587.30   54,065.31   53,286.69   53,819.61
2026-03-16     53,627.86   53,983.51   53,113.95   53,751.

In [ ]:
# ─── デバッグ時に使用: ページソースとグラフ要素の確認 ─────────────────────
debug_options = Options()
debug_options.add_argument('--headless')
debug_options.add_argument('--no-sandbox')
debug_options.add_argument('--disable-dev-shm-usage')
debug_options.add_argument('--window-size=1920,1080')
debug_options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/124.0.0.0 Safari/537.36'
)

debug_service = Service(ChromeDriverManager().install())
debug_driver  = webdriver.Chrome(service=debug_service, options=debug_options)

try:
    debug_driver.get('https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month')
    time.sleep(5)  # JS描画待ち

    page_html = debug_driver.page_source
    soup_debug = BeautifulSoup(page_html, 'html.parser')

    # 「chart」「data」「value」「date」を含むクラス名を持つ要素を列挙
    keywords = ['chart', 'data', 'value', 'date', 'price']
    found_classes = set()
    for tag in soup_debug.find_all(True):
        cls = ' '.join(tag.get('class', []))
        if any(kw in cls.lower() for kw in keywords):
            found_classes.add(f'<{tag.name} class="{cls}">')

    print('=== 関連クラスを持つ要素 ===')
    for c in sorted(found_classes):
        print(c)

    # ページソースの先頭 3000 文字を確認
    print('\n=== ページソース（先頭6000字） ===')
    print(page_html[:6000])

finally:
    debug_driver.quit()